In [2]:

# 3️⃣ Load dataset
import pandas as pd

data_path = "/Users/parthchaudhary/Desktop/MS Analytics Sem 4/Project/project/online_shoppers_intention.csv"
df = pd.read_csv(data_path)

print("Dataset loaded!")
display(df.head())

# 4️⃣ Clean data
df = df.dropna()
print("After cleaning:")
display(df.head())

# 5️⃣ Convert rows → text chunks
text_chunks = []

for _, row in df.iterrows():
    text = (
        f"Month: {row['Month']}, Visitor Type: {row['VisitorType']}, Weekend: {row['Weekend']}, "
        f"Revenue: {row['Revenue']}, Administrative: {row['Administrative']}, "
        f"Administrative Duration: {row['Administrative_Duration']}, "
        f"Informational: {row['Informational']}, "
        f"Informational Duration: {row['Informational_Duration']}, "
        f"ProductRelated: {row['ProductRelated']}, "
        f"ProductRelated Duration: {row['ProductRelated_Duration']}, "
        f"Bounce Rates: {row['BounceRates']}, Exit Rates: {row['ExitRates']}, "
        f"Page Values: {row['PageValues']}, Traffic Type: {row['TrafficType']}, "
        f"Region: {row['Region']}, Browser: {row['Browser']}, "
        f"Operating System: {row['OperatingSystems']}"
    )
    text_chunks.append(text)

print(f"Total chunks: {len(text_chunks)}")
print("Sample chunk:\n", text_chunks[0])

# 6️⃣ Load FREE embedding model
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')  # 🔥 FREE + fast + good

# 7️⃣ Generate embeddings (FAST batch)
embeddings = model.encode(
    text_chunks,
    batch_size=256,
    show_progress_bar=True
)

print("Embeddings generated!")

# 8️⃣ Build FAISS index
import numpy as np
import faiss

embeddings = np.array(embeddings).astype('float32')

dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

print(f"FAISS index created with {index.ntotal} vectors")

# 9️⃣ Save index
import os

vector_path = "/Users/parthchaudhary/Desktop/MS Analytics Sem 4/Project/project/online_shoppers_intention.csv"
os.makedirs(os.path.dirname(vector_path), exist_ok=True)

faiss.write_index(index, vector_path)

print(f"Vector DB saved at {vector_path}")

Dataset loaded!


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


After cleaning:


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


Total chunks: 12330
Sample chunk:
 Month: Feb, Visitor Type: Returning_Visitor, Weekend: False, Revenue: False, Administrative: 0, Administrative Duration: 0.0, Informational: 0, Informational Duration: 0.0, ProductRelated: 1, ProductRelated Duration: 0.0, Bounce Rates: 0.2, Exit Rates: 0.2, Page Values: 0.0, Traffic Type: 1, Region: 1, Browser: 1, Operating System: 1


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8028.05it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 49/49 [00:23<00:00,  2.10it/s]

Embeddings generated!
FAISS index created with 12330 vectors
Vector DB saved at /Users/parthchaudhary/Desktop/MS Analytics Sem 4/Project/project/online_shoppers_intention.csv


User Question → Convert to embedding → Search FAISS → Return relevant rows

In [3]:
# =============================================
# 🔟 RAG Retrieval System (QUERY ENGINE)
# =============================================

# Function to search similar data
def search(query, k=5):
    # Convert query → embedding
    query_embedding = model.encode([query])

    # Search in FAISS
    D, I = index.search(
        np.array(query_embedding).astype('float32'), k
    )

    # Get matching rows
    results = [text_chunks[i] for i in I[0]]

    return results


# =============================================
# 🔥 TEST YOUR SYSTEM
# =============================================

query = "Users who generate revenue and have high page values"

results = search(query, k=5)

print("🔍 Top Matching Results:\n")

for i, r in enumerate(results):
    print(f"{i+1}. {r}\n")

🔍 Top Matching Results:

1. Month: Aug, Visitor Type: New_Visitor, Weekend: True, Revenue: False, Administrative: 1, Administrative Duration: 250.8, Informational: 2, Informational Duration: 194.4, ProductRelated: 2, ProductRelated Duration: 194.4, Bounce Rates: 0.0, Exit Rates: 0.05, Page Values: 0.0, Traffic Type: 6, Region: 1, Browser: 2, Operating System: 2

2. Month: Aug, Visitor Type: New_Visitor, Weekend: False, Revenue: False, Administrative: 6, Administrative Duration: 265.4, Informational: 0, Informational Duration: 0.0, ProductRelated: 6, ProductRelated Duration: 161.6, Bounce Rates: 0.0, Exit Rates: 0.022222222, Page Values: 0.0, Traffic Type: 3, Region: 3, Browser: 1, Operating System: 1

3. Month: Oct, Visitor Type: New_Visitor, Weekend: True, Revenue: False, Administrative: 5, Administrative Duration: 163.0, Informational: 0, Informational Duration: 0.0, ProductRelated: 5, ProductRelated Duration: 25.0, Bounce Rates: 0.02, Exit Rates: 0.04, Page Values: 0.0, Traffic Type

In [4]:
def answer_question(query):
    results = search(query, k=5)

    answer = "Based on retrieved data:\n\n"

    for r in results:
        answer += "- " + r + "\n"

    return answer


print(answer_question("Which users are likely to generate revenue?"))

Based on retrieved data:

- Month: Aug, Visitor Type: New_Visitor, Weekend: True, Revenue: False, Administrative: 1, Administrative Duration: 250.8, Informational: 2, Informational Duration: 194.4, ProductRelated: 2, ProductRelated Duration: 194.4, Bounce Rates: 0.0, Exit Rates: 0.05, Page Values: 0.0, Traffic Type: 6, Region: 1, Browser: 2, Operating System: 2
- Month: Aug, Visitor Type: New_Visitor, Weekend: True, Revenue: False, Administrative: 5, Administrative Duration: 82.6, Informational: 1, Informational Duration: 14.2, ProductRelated: 8, ProductRelated Duration: 66.53333333, Bounce Rates: 0.0, Exit Rates: 0.02, Page Values: 0.0, Traffic Type: 1, Region: 3, Browser: 1, Operating System: 4
- Month: May, Visitor Type: New_Visitor, Weekend: True, Revenue: False, Administrative: 6, Administrative Duration: 79.5, Informational: 1, Informational Duration: 19.0, ProductRelated: 11, ProductRelated Duration: 314.25, Bounce Rates: 0.0, Exit Rates: 0.003333333, Page Values: 0.0, Traffic T

In [5]:
def analyze_results(results):
    insights = []

    revenue_count = 0
    returning_visitors = 0
    high_page_values = 0

    for r in results:
        if "Revenue: True" in r:
            revenue_count += 1
        if "Returning_Visitor" in r:
            returning_visitors += 1
        if "Page Values: 0.0" not in r:
            high_page_values += 1

    insights.append(f"{revenue_count}/{len(results)} users generated revenue")
    insights.append(f"{returning_visitors}/{len(results)} are returning visitors")
    insights.append(f"{high_page_values}/{len(results)} have non-zero page values")

    return insights


def answer(query):
    results = search(query, k=5)

    print("🔍 Retrieved Data:\n")
    for r in results:
        print("-", r)

    print("\n📊 Insights:\n")
    insights = analyze_results(results)
    for i in insights:
        print("-", i)

In [6]:
import requests

def call_llm(prompt, model="llama3"):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": model,
            "prompt": prompt,
            "stream": False
        }
    )
    return response.json()["response"]

In [7]:
def analyst_agent(query, prompt_style, model="llama3"):
    
    results = search(query, k=5)
    context = "\n".join(results)
    
    prompt = f"""
    {prompt_style}

    You are analyzing user behavior data.

    Context:
    {context}

    Question:
    {query}
    """
    
    response = call_llm(prompt, model=model)
    
    return response

In [8]:
PROMPTS = [
    "Give business insights clearly.",
    "Act as a senior data analyst and explain patterns.",
    "Provide bullet-point insights.",
    "Write an executive summary for stakeholders."
]

In [9]:
MODELS = ["llama3", "mistral", "phi3", "gemma"]

In [10]:
queries = [
    "Which users generate revenue?",
    "Do returning visitors generate more revenue?",
    "Does page value impact revenue?",
    "How does traffic type affect conversions?",
    "What patterns exist in weekend visitors?"
]

In [11]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13510.13it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
def search(query, k=5):
    query_embedding = embedding_model.encode([query])

    D, I = index.search(
        np.array(query_embedding).astype('float32'), k
    )

    results = [text_chunks[i] for i in I[0]]

    return results

In [13]:
import requests

def call_llm(prompt, model="llama3"):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": model,
            "prompt": prompt,
            "stream": False
        }
    )
    
    return response.json()["response"]

In [14]:
def analyst_agent(query, prompt_style, llm_model="llama3"):
    
    results = search(query, k=5)
    context = "\n".join(results)
    
    prompt = f"""
    {prompt_style}

    You are analyzing online shopping behavior data.

    Context:
    {context}

    Question:
    {query}

    Provide meaningful business insights.
    """
    
    response = call_llm(prompt, model=llm_model)
    
    return response

In [15]:
PROMPTS = [
    "Give business insights clearly.",
    "Act as a senior data analyst and explain patterns.",
    "Provide bullet-point insights.",
    "Write an executive summary for stakeholders."
]

In [16]:
MODELS = ["llama3", "mistral", "phi3", "gemma"]

In [17]:
queries = [
    "Which users generate revenue?",
    "Do returning visitors generate more revenue?",
    "Does page value impact revenue?",
    "How does traffic type affect conversions?",
    "What patterns exist in weekend visitors?"
]

In [22]:
# ================================
# FULL PIPELINE (ONE CELL)
# ================================

import requests
import pandas as pd

# -------------------------------
# 1. LLM CALL (SAFE + ROBUST)
# -------------------------------
def call_llm(prompt, model="llama3"):
    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": model,
                "prompt": prompt,
                "stream": False
            }
        )
        
        res_json = response.json()

        if "response" in res_json:
            return res_json["response"]
        else:
            print("❌ ERROR from Ollama:", res_json)
            return "ERROR"

    except Exception as e:
        print("❌ Exception:", e)
        return "ERROR"


# -------------------------------
# 2. ANALYST AGENT
# -------------------------------
def analyst_agent(query, prompt_style, llm_model="llama3"):
    
    prompt = f"""
    {prompt_style}

    Question:
    {query}

    Provide clear, structured, and meaningful business insights.
    """
    
    return call_llm(prompt, model=llm_model)


# -------------------------------
# 3. CONFIG
# -------------------------------
MODELS = ["llama3"]   # ✅ safe for now

PROMPTS = [
    "Give business insights based on the data.",
    "Act as a senior data analyst and provide detailed reasoning.",
    "Provide bullet-point insights with reasoning.",
    "Write an executive summary with key findings."
]

# Replace with your actual queries if needed
queries = [
    "What factors influence customer purchase?",
    "How does weekend affect revenue?",
    "Which user type generates more revenue?",
    "What patterns can you observe in customer behavior?",
    "What recommendations can you give to improve sales?"
]


# -------------------------------
# 4. RUN EVALUATION
# -------------------------------
evaluation_results = []

for model_name in MODELS:
    for prompt_style in PROMPTS:
        for q in queries:
            
            print(f"Running → Model: {model_name}, Prompt: {prompt_style[:30]}...")
            
            response = analyst_agent(
                query=q,
                prompt_style=prompt_style,
                llm_model=model_name
            )
            
            evaluation_results.append({
                "model": model_name,
                "prompt": prompt_style,
                "query": q,
                "response": response
            })


# -------------------------------
# 5. SAVE RESULTS
# -------------------------------
df_results = pd.DataFrame(evaluation_results)

print("\n✅ DONE! Sample Output:\n")
display(df_results.head())


# -------------------------------
# 6. (OPTIONAL) SAVE CSV
# -------------------------------
df_results.to_csv("evaluation_results.csv", index=False)

print("\n📁 Saved as evaluation_results.csv")

Running → Model: llama3, Prompt: Give business insights based o...
Running → Model: llama3, Prompt: Give business insights based o...
Running → Model: llama3, Prompt: Give business insights based o...
Running → Model: llama3, Prompt: Give business insights based o...
Running → Model: llama3, Prompt: Give business insights based o...
Running → Model: llama3, Prompt: Act as a senior data analyst a...
Running → Model: llama3, Prompt: Act as a senior data analyst a...
Running → Model: llama3, Prompt: Act as a senior data analyst a...
Running → Model: llama3, Prompt: Act as a senior data analyst a...
Running → Model: llama3, Prompt: Act as a senior data analyst a...
Running → Model: llama3, Prompt: Provide bullet-point insights ...
Running → Model: llama3, Prompt: Provide bullet-point insights ...
Running → Model: llama3, Prompt: Provide bullet-point insights ...
Running → Model: llama3, Prompt: Provide bullet-point insights ...
Running → Model: llama3, Prompt: Provide bullet-point insights

,model,prompt,query,response
0,llama3,Give business insights based on the data.,What factors influence customer purchase?,**Business Insights: Factors Influencing Custo...
1,llama3,Give business insights based on the data.,How does weekend affect revenue?,**Business Insight: The Impact of Weekends on ...
2,llama3,Give business insights based on the data.,Which user type generates more revenue?,**Business Insights: User Type Revenue Analysi...
3,llama3,Give business insights based on the data.,What patterns can you observe in customer beha...,"Based on the provided data, I've identified so..."
4,llama3,Give business insights based on the data.,What recommendations can you give to improve s...,"Based on the available data, I've identified s..."



📁 Saved as evaluation_results.csv
